# Sim-to-Real Gap в RL для шагающих роботов

Воспроизведение трёх источников разрыва на HalfCheetah-v4:
1. Трение (Transition gap)
2. Задержка действий (Action gap)
3. Шум сенсоров (Observation gap)

In [1]:
!pip install gymnasium[mujoco]
!pip install stable-baselines3[extra]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 3.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
env = gym.make("HalfCheetah-v4")          # Обучение badeline
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=100_000)
model.save("baseline_halfcheetah")
print("Модель сохранена")

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment HalfCheetah-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -323     |
| time/              |          |
|    fps             | 1025     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -311        |
| time/                   |             |
|    fps                  | 694         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010627718 |
|    clip_fraction        | 0.109       |
|    clip_range           | 0.2         |
|    entropy_loss         | -8.52       |
|    explained_variance   | -0.0158     |
|    learning_rate        | 0.0003      |
|    loss                 | 27.9        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0211     |
|    std                  | 1           |
|    value_loss           | 55.8        |
-----------------------------------------
-----------------------------------------
| rollout/                |       

In [4]:
mean_baseline, std_baseline = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Baseline: {mean_baseline:.2f} +/- {std_baseline:.2f}")

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Baseline: 344.69 +/- 149.24


In [12]:
class FrictionWrapper(gym.Wrapper):
    def __init__(self, env, friction_multiplier=1.0):
        super().__init__(env)
        self.friction_multiplier = friction_multiplier

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        model_mj = self.env.unwrapped.model
        floor_id = model_mj.geom("floor").id
        original = model_mj.geom_friction[floor_id][0]
        model_mj.geom_friction[floor_id][0] = original * self.friction_multiplier
        return obs, info


class ActionDelayWrapper(gym.Wrapper):
    def __init__(self, env, delay=0):
        super().__init__(env)
        self.delay = delay
        self.action_buffer = []

    def reset(self, **kwargs):
        self.action_buffer = []
        return self.env.reset(**kwargs)

    def step(self, action):
        self.action_buffer.append(action)
        if len(self.action_buffer) <= self.delay:
            delayed_action = self.action_buffer[0]
        else:
            delayed_action = self.action_buffer.pop(0)
        return self.env.step(delayed_action)


class SensorNoiseWrapper(gym.ObservationWrapper):
    def __init__(self, env, noise_std=0.0):
        super().__init__(env)
        self.noise_std = noise_std

    def observation(self, obs):
        noise = np.random.normal(0, self.noise_std, size=obs.shape)
        return obs + noise

In [6]:
def sweep(model, wrapper_class, values, param_name):
    results = []
    for value in values:
        env = gym.make("HalfCheetah-v4")
        env = wrapper_class(env, **{param_name: value})
        mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=10)
        results.append((value, mean_r, std_r))
        print(f"{param_name}={value}: reward = {mean_r:.2f} +/- {std_r:.2f}")
    return results

In [7]:
friction_results = sweep(
    model, FrictionWrapper,
    [0.3, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0],
    "friction_multiplier"
)

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment HalfCheetah-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


friction_multiplier=0.3: reward = 400.98 +/- 57.61
friction_multiplier=0.5: reward = 389.99 +/- 122.63
friction_multiplier=0.8: reward = 398.72 +/- 66.09
friction_multiplier=1.0: reward = 404.76 +/- 43.68
friction_multiplier=1.2: reward = 72.71 +/- 229.13
friction_multiplier=1.5: reward = -48.94 +/- 157.51
friction_multiplier=2.0: reward = -73.61 +/- 114.27
friction_multiplier=3.0: reward = -93.88 +/- 167.93


In [8]:
delay_results = sweep(
    model, ActionDelayWrapper,
    [0, 1, 2, 3, 4, 5],
    "delay"
)

delay=0: reward = 371.83 +/- 78.36
delay=1: reward = 219.51 +/- 132.00
delay=2: reward = 317.78 +/- 91.78
delay=3: reward = 294.08 +/- 38.47
delay=4: reward = 506.77 +/- 82.92
delay=5: reward = 516.85 +/- 156.97


In [ ]:
noise_results = sweep(
    model, SensorNoiseWrapper,
    [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0],
    "noise_std"
)

noise_std=0.0: reward = 438.93 +/- 58.71
noise_std=0.05: reward = 466.80 +/- 65.49
noise_std=0.1: reward = 400.61 +/- 107.71
noise_std=0.2: reward = 347.16 +/- 61.71
noise_std=0.3: reward = 419.15 +/- 83.85
noise_std=0.5: reward = 416.14 +/- 71.73
noise_std=0.8: reward = 374.81 +/- 74.61


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

f_x = [r[0] for r in friction_results]
f_y = [r[1] for r in friction_results]
axes[0].plot(f_x, f_y, marker="o", linewidth=2, color="#C44E52")
axes[0].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Множитель трения")
axes[0].set_ylabel("Reward")
axes[0].set_title("Трение", fontweight="bold")
axes[0].grid(alpha=0.3)

d_x = [r[0] for r in delay_results]
d_y = [r[1] for r in delay_results]
axes[1].plot(d_x, d_y, marker="o", linewidth=2, color="#C44E52")
axes[1].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Задержка действий (шаги)")
axes[1].set_ylabel("Reward")
axes[1].set_title("Задержка действий", fontweight="bold")
axes[1].grid(alpha=0.3)

n_x = [r[0] for r in noise_results]
n_y = [r[1] for r in noise_results]
axes[2].plot(n_x, n_y, marker="o", linewidth=2, color="#55A868")
axes[2].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
axes[2].set_xlabel("Стандартное отклонение шума")
axes[2].set_ylabel("Reward")
axes[2].set_title("Шум сенсоров", fontweight="bold")
axes[2].grid(alpha=0.3)

plt.suptitle("Деградация политики при различных источниках sim-to-real разрыва", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("sweep_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [13]:
with open("results.txt", "w") as f:
    f.write(f"Baseline: {mean_baseline:.2f} +/- {std_baseline:.2f}\n\n")
    f.write("=== Friction sweep ===\n")
    for value, mean_r, std_r in friction_results:
        f.write(f"{value}: {mean_r:.2f} +/- {std_r:.2f}\n")
    f.write("\n=== Delay sweep ===\n")
    for value, mean_r, std_r in delay_results:
        f.write(f"{value}: {mean_r:.2f} +/- {std_r:.2f}\n")
    f.write("\n=== Noise sweep ===\n")
    for value, mean_r, std_r in noise_results:
        f.write(f"{value}: {mean_r:.2f} +/- {std_r:.2f}\n")
print("Результаты сохранены")

Результаты сохранены
